<a href="https://colab.research.google.com/github/ananthakrishnanm010/Emotion_NLP_Project/blob/main/NLP_Parallel_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Libraries

In [43]:
import pandas as pd
import numpy as np
import string
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
from nltk.stem import  WordNetLemmatizer
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

# Read Dataset

In [3]:
df_emotion = pd.read_parquet("hf://datasets/dair-ai/emotion/unsplit/train-00000-of-00001.parquet")


df_emotion.head(3)

,text,label
0,i feel awful about it too because it s my job ...,0
1,im alone i feel awful,0
2,ive probably mentioned this before but i reall...,1


# EDA

In [8]:
df_emotion.head()

,text,label
0,i feel awful about it too because it s my job ...,0
1,im alone i feel awful,0
2,ive probably mentioned this before but i reall...,1
3,i was feeling a little low few days back,0
4,i beleive that i am much more sensitive to oth...,2


In [5]:
df_emotion.shape

(416809, 2)

In [6]:
df_emotion.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 416809 entries, 0 to 416808
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    416809 non-null  object
 1   label   416809 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 6.4+ MB


In [7]:
df_emotion.isnull().sum()

,0
text,0
label,0


In [12]:
df_emotion['label'].value_counts()

,count
label,
1,141067
0,121187
3,57317
4,47712
2,34554
5,14972


In [ ]:
# inference
# The target column is 'label'
# Based on the text in the 'text' column we have to identify the emotion in label
# label:  sadness (0), joy (1), love (2), anger (3), fear (4), surprise (5).

In [21]:
df_emotion = df_emotion[['label','text']]  # to display lablel first and text later for convinience

# Preprocessing

## Duplicate removal

In [23]:
df_emotion.duplicated().sum()

np.int64(0)

In [24]:
df_emotion = df_emotion.drop_duplicates()

In [25]:
df_emotion.duplicated().sum()

np.int64(0)

## Punctuation Removal

In [26]:
def remove_punc(text):
  text = str(text)
  text = text.replace("�", "")
  punctuationless_text = ''.join(i for i in text if i not in string.punctuation)
  return punctuationless_text

In [27]:
df_emotion["punctuationless_text"] = [remove_punc(text) for text in df_emotion["text"]]

In [28]:
df_emotion.head()

,label,text,punctuationless_text
0,0,i feel awful about it too because it s my job ...,i feel awful about it too because it s my job ...
1,0,im alone i feel awful,im alone i feel awful
2,1,ive probably mentioned this before but i reall...,ive probably mentioned this before but i reall...
3,0,i was feeling a little low few days back,i was feeling a little low few days back
4,2,i beleive that i am much more sensitive to oth...,i beleive that i am much more sensitive to oth...


## Lowercasing

In [29]:
df_emotion["lowercased"] = [text.lower() for text in df_emotion["punctuationless_text"]]

In [30]:
df_emotion.head()

,label,text,punctuationless_text,lowercased
0,0,i feel awful about it too because it s my job ...,i feel awful about it too because it s my job ...,i feel awful about it too because it s my job ...
1,0,im alone i feel awful,im alone i feel awful,im alone i feel awful
2,1,ive probably mentioned this before but i reall...,ive probably mentioned this before but i reall...,ive probably mentioned this before but i reall...
3,0,i was feeling a little low few days back,i was feeling a little low few days back,i was feeling a little low few days back
4,2,i beleive that i am much more sensitive to oth...,i beleive that i am much more sensitive to oth...,i beleive that i am much more sensitive to oth...


## Tokenization

In [31]:
# we convert the entire text into a list of unique words.
def tokenization(text):
  word_list = nltk.word_tokenize(text)
  return word_list

In [34]:
df_emotion["tokenized"] = [tokenization(text) for text in df_emotion["lowercased"]]

In [35]:
df_emotion.head()

,label,text,punctuationless_text,lowercased,tokenized
0,0,i feel awful about it too because it s my job ...,i feel awful about it too because it s my job ...,i feel awful about it too because it s my job ...,"[i, feel, awful, about, it, too, because, it, ..."
1,0,im alone i feel awful,im alone i feel awful,im alone i feel awful,"[im, alone, i, feel, awful]"
2,1,ive probably mentioned this before but i reall...,ive probably mentioned this before but i reall...,ive probably mentioned this before but i reall...,"[ive, probably, mentioned, this, before, but, ..."
3,0,i was feeling a little low few days back,i was feeling a little low few days back,i was feeling a little low few days back,"[i, was, feeling, a, little, low, few, days, b..."
4,2,i beleive that i am much more sensitive to oth...,i beleive that i am much more sensitive to oth...,i beleive that i am much more sensitive to oth...,"[i, beleive, that, i, am, much, more, sensitiv..."


## Stop Words Removal

In [37]:
stop_words_list = nltk.corpus.stopwords.words('english')
stop_words_list
def remove_stopwords(tokens):
  stop_word_less_list = [i for i in tokens if i not in stop_words_list]
  return stop_word_less_list

In [38]:
df_emotion["stop_words_removed"] = [remove_stopwords(text) for text in df_emotion["tokenized"]]

In [39]:
df_emotion.head()

,label,text,punctuationless_text,lowercased,tokenized,stop_words_removed
0,0,i feel awful about it too because it s my job ...,i feel awful about it too because it s my job ...,i feel awful about it too because it s my job ...,"[i, feel, awful, about, it, too, because, it, ...","[feel, awful, job, get, position, succeed, hap..."
1,0,im alone i feel awful,im alone i feel awful,im alone i feel awful,"[im, alone, i, feel, awful]","[im, alone, feel, awful]"
2,1,ive probably mentioned this before but i reall...,ive probably mentioned this before but i reall...,ive probably mentioned this before but i reall...,"[ive, probably, mentioned, this, before, but, ...","[ive, probably, mentioned, really, feel, proud..."
3,0,i was feeling a little low few days back,i was feeling a little low few days back,i was feeling a little low few days back,"[i, was, feeling, a, little, low, few, days, b...","[feeling, little, low, days, back]"
4,2,i beleive that i am much more sensitive to oth...,i beleive that i am much more sensitive to oth...,i beleive that i am much more sensitive to oth...,"[i, beleive, that, i, am, much, more, sensitiv...","[beleive, much, sensitive, peoples, feelings, ..."


## Lemmatization

In [41]:
lem_obj = WordNetLemmatizer()
def lemmatizing(stem_token_list):
  lem_list = [lem_obj.lemmatize(word) for word in stem_token_list ]
  return lem_list


In [44]:
df_emotion["lemmatized_text"] = [lemmatizing(text) for text in df_emotion["stop_words_removed"]]

In [45]:
df_emotion.head()

,label,text,punctuationless_text,lowercased,tokenized,stop_words_removed,lemmatized_text
0,0,i feel awful about it too because it s my job ...,i feel awful about it too because it s my job ...,i feel awful about it too because it s my job ...,"[i, feel, awful, about, it, too, because, it, ...","[feel, awful, job, get, position, succeed, hap...","[feel, awful, job, get, position, succeed, hap..."
1,0,im alone i feel awful,im alone i feel awful,im alone i feel awful,"[im, alone, i, feel, awful]","[im, alone, feel, awful]","[im, alone, feel, awful]"
2,1,ive probably mentioned this before but i reall...,ive probably mentioned this before but i reall...,ive probably mentioned this before but i reall...,"[ive, probably, mentioned, this, before, but, ...","[ive, probably, mentioned, really, feel, proud...","[ive, probably, mentioned, really, feel, proud..."
3,0,i was feeling a little low few days back,i was feeling a little low few days back,i was feeling a little low few days back,"[i, was, feeling, a, little, low, few, days, b...","[feeling, little, low, days, back]","[feeling, little, low, day, back]"
4,2,i beleive that i am much more sensitive to oth...,i beleive that i am much more sensitive to oth...,i beleive that i am much more sensitive to oth...,"[i, beleive, that, i, am, much, more, sensitiv...","[beleive, much, sensitive, peoples, feelings, ...","[beleive, much, sensitive, people, feeling, te..."


# Embedding

## TF-IDF  

# Model